In [278]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb
from sklearn.metrics import f1_score


In [279]:
# Loading the treated data
train_dataset = pd.read_csv('data/train_dataset_treated.csv')

# Loading the control dataset
control_dataset = pd.read_csv('data/train_radiomics_occipital_CONTROL.csv')

In [280]:
# Guaranteeing the same columns in both datasets
control_dataset = control_dataset[train_dataset.columns]

In [281]:
# --- Using encoding to transform the categorical columns into numerical columns
replace_map = {'Transition': {'CN-CN': 0, 'AD-AD': 1, 'CN-MCI': 2, 'MCI-AD': 3, 'MCI-MCI': 4}}
control_dataset.replace(replace_map, inplace=True)
control_dataset.replace(replace_map, inplace=True)

C:\Users\Utilizador\AppData\Local\Temp\ipykernel_16516\1480064811.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  control_dataset.replace(replace_map, inplace=True)


In [282]:
# Defining the function that will use the model to complete the missing values
# Using the model to complete the test dataset

def complete_dataset(model_name, model):

    test_dataset = pd.read_csv('data/test_dataset_treated.csv')

    test_pred = model.predict(test_dataset)
    test_dataset['Transition'] = test_pred
    test_dataset.head()

    # Dropping all columns but the Transition column
    test_dataset.drop(test_dataset.columns.difference(['Transition']), axis=1, inplace=True)

    # Creating a RowId column to store the index, starting from 1
    test_dataset['RowId'] = np.arange(1, test_dataset.shape[0] + 1)

    # Placing the RowId column in the first position
    cols = test_dataset.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    test_dataset = test_dataset[cols]

    # Transforming the Transition column back to its original values
    replace_map = {'Transition': {0: 'CN-CN', 1: 'AD-AD', 2: 'CN-MCI', 3: 'MCI-AD', 4: 'MCI-MCI'}}
    test_dataset.replace(replace_map, inplace=True)
    test_dataset.head()

    # Saving the test dataset to a csv file
    test_dataset.to_csv('test_predictions_' + model_name + '.csv', index=False)

## Random Forest

In [283]:
# Running a Random Forest Classifier
#model_name = 'random_forest'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

rfc = RandomForestClassifier(n_estimators=100, random_state=123)
rfc.fit(X_train, y_train)
rfc_pred = rfc.predict(X_test)

# Printing f1 score
print("f1_score for the test data: ")
print(f1_score(y_test, rfc_pred, average='weighted'))

# Running for the control dataset as well
X_control = control_dataset.drop('Transition', axis=1)
y_control = control_dataset['Transition']

control_pred = rfc.predict(X_control)
print("f1_score for the control data: ")
print(f1_score(y_control, control_pred, average='weighted'))

f1_score for the test data: 
0.4480264648718829
f1_score for the control data: 
0.21975766360256407


In [284]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, rfc_pred))

Confusion matrix
[[ 8  2  1  3  5]
 [ 0  8  0  6  1]
 [ 0  0 18  0  0]
 [ 4  6  1  5  0]
 [10  2  0  7  3]]


In [285]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, rfc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.36      0.42      0.39        19
           1       0.44      0.53      0.48        15
           2       0.90      1.00      0.95        18
           3       0.24      0.31      0.27        16
           4       0.33      0.14      0.19        22

    accuracy                           0.47        90
   macro avg       0.46      0.48      0.46        90
weighted avg       0.45      0.47      0.45        90



In [286]:
# Completing the test dataset
complete_dataset('random_forest', rfc)

## XGBoost

In [287]:
#model_name = 'xgboost'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=123)

xgbc = xgb.XGBClassifier(max_depth = 1, objective='req:squarederror', random_state=123, learning_rate=0.2, n_estimators=52)
xgbc.fit(X_train, y_train)
xgbc_pred = xgbc.predict(X_test)

# Printing f1 score
print("f1_score for the test data: ")
print(f1_score(y_test, xgbc_pred, average='weighted'))

# Running for the control dataset as well
X_control = control_dataset.drop('Transition', axis=1)
y_control = control_dataset['Transition']

control_pred = xgbc.predict(X_control)
print("f1_score for the control data: ")
print(f1_score(y_control, control_pred, average='weighted'))

f1_score for the test data: 
0.43258227267515187
f1_score for the control data: 
0.17708324462422823


In [288]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, xgbc_pred))

Confusion matrix
[[ 7  3  1  4  4]
 [ 0  9  0  5  1]
 [ 2  0 16  0  0]
 [ 0  6  1  4  5]
 [ 8  1  2  7  4]]


In [289]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, xgbc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.41      0.37      0.39        19
           1       0.47      0.60      0.53        15
           2       0.80      0.89      0.84        18
           3       0.20      0.25      0.22        16
           4       0.29      0.18      0.22        22

    accuracy                           0.44        90
   macro avg       0.43      0.46      0.44        90
weighted avg       0.43      0.44      0.43        90



In [290]:
# Completing the test dataset
complete_dataset('xgboost', xgbc)